# Approval Model Walkthrough

This notebook explores the synthetic dataset and the PyTorch model behind **Stage 2 (The Predictor)** of the Financial Decision Engine.

1. Generate and inspect the synthetic applications
2. Check the patterns a lender would expect
3. Train the neural network and compare it with a logistic regression baseline
4. Ask the trained model "what if?" questions
5. See the tensor-shape gotcha in action

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # make the project packages importable

import pandas as pd
import torch

from backend.model import build_features
from backend.predictor import ApprovalPredictor
from ml.generate_data import generate_applications
from ml.train import train_model

pd.set_option("display.float_format", "{:,.2f}".format)

## 1. The synthetic dataset

Each row is one applicant applying for one card. `card_tier` runs from 0 (Starter) to 3 (Elite). `approved` is the outcome produced by a hidden "bank policy" plus a weighted coin flip.

In [2]:
df = generate_applications(rows=5_000, seed=42)
df["debt_to_income"] = df["existing_debt"] / df["annual_income"]
print(f"{len(df):,} applications, overall approval rate {df['approved'].mean():.1%}")
df.head()

5,000 applications, overall approval rate 55.4%


,credit_score,annual_income,existing_debt,card_tier,approved,debt_to_income
0,706,"52,100.00","17,800.00",0,1,0.34
1,592,"50,900.00","4,800.00",2,0,0.09
2,744,"40,900.00","1,900.00",3,1,0.05
3,760,"75,900.00",0.00,1,1,0.00
4,514,"68,300.00","61,100.00",0,0,0.89


In [3]:
df.describe()

,credit_score,annual_income,existing_debt,card_tier,approved,debt_to_income
count,"5,000.00","5,000.00","5,000.00","5,000.00","5,000.00","5,000.00"
mean,678.18,"70,043.70","24,000.98",1.52,0.55,0.34
std,84.58,"42,266.38","29,561.10",1.10,0.50,0.32
min,370.00,"12,000.00",0.00,0.00,0.00,0.00
25%,621.00,"41,700.00","5,375.00",1.00,0.00,0.11
50%,680.00,"59,300.00","14,600.00",2.00,1.00,0.26
75%,734.00,"86,825.00","31,800.00",2.00,1.00,0.48
max,900.00,"500,000.00","357,900.00",3.00,1.00,2.69


## 2. Do the patterns make sense?

Approval rate by credit score band (rows) and card tier (columns). Rates should rise with score and fall with tier.

In [4]:
score_band = pd.cut(df["credit_score"], [299, 560, 620, 680, 740, 900])
df.pivot_table(index=score_band, columns="card_tier", values="approved", aggfunc="mean", observed=True)

card_tier,0,1,2,3
credit_score,,,,
"(299, 560]",0.34,0.15,0.03,0.02
"(560, 620]",0.57,0.33,0.21,0.08
"(620, 680]",0.75,0.63,0.38,0.17
"(680, 740]",0.86,0.74,0.60,0.41
"(740, 900]",0.96,0.94,0.84,0.71


Approval rate by debt-to-income ratio. Owing more relative to what you earn should hurt.

In [5]:
dti_band = pd.cut(df["debt_to_income"], [-0.01, 0.1, 0.2, 0.4, 0.6, 1.0, 10])
df.groupby(dti_band, observed=True)["approved"].agg(["mean", "count"])

,mean,count
debt_to_income,,
"(-0.01, 0.1]",0.69,1183
"(0.1, 0.2]",0.62,844
"(0.2, 0.4]",0.57,1345
"(0.4, 0.6]",0.49,760
"(0.6, 1.0]",0.39,635
"(1.0, 10.0]",0.20,233


## 3. Train the network

`train_model()` is the same function behind `python -m ml.train`. Here it saves to `models/notebook_model.pt` (ignored by git) so the committed `approval_mlp.pt` is left untouched.

In [6]:
model_path = Path.cwd().parent / "models" / "notebook_model.pt"
metrics = train_model(rows=5_000, epochs=60, output_path=model_path)

Data: 4,000 train / 1,000 validation rows, approval rate 55.4%


  epoch   1/60  validation loss 0.5916


  epoch  10/60  validation loss 0.4112


  epoch  20/60  validation loss 0.4119


  epoch  30/60  validation loss 0.4107


  epoch  40/60  validation loss 0.4081


  epoch  50/60  validation loss 0.4098


  epoch  60/60  validation loss 0.4080

Best epoch: 59  (validation loss 0.4063)
Neural network  -> accuracy 82.2%, ROC-AUC 0.895
LogReg baseline -> accuracy 78.8%, ROC-AUC 0.873

Sample predictions for a Standard (tier 1) card:
  Strong   (score 780, $95k income, $5k debt): 98%
  Average  (score 680, $55k income, $15k debt): 80%
  Stretched(score 590, $32k income, $28k debt): 6%

Saved model to models/notebook_model.pt


In [7]:
pd.DataFrame(
    {
        "Neural network (MLP)": [metrics["mlp_val_accuracy"], metrics["mlp_val_auc"]],
        "Logistic regression": [metrics["logreg_val_accuracy"], metrics["logreg_val_auc"]],
    },
    index=["Validation accuracy", "Validation ROC-AUC"],
).map("{:.3f}".format)

,Neural network (MLP),Logistic regression
Validation accuracy,0.822,0.788
Validation ROC-AUC,0.895,0.873


## 4. "What if?" questions

Approval probability for someone earning \$60k with \$12k of debt, as their credit score changes, for each card tier.

In [8]:
predictor = ApprovalPredictor(model_path)
tiers = {0: "Starter", 1: "Standard", 2: "Premium", 3: "Elite"}

what_if = pd.DataFrame(
    {score: predictor.predict(score, 60_000, 12_000, card_tiers=list(tiers)) for score in range(500, 851, 50)},
    index=tiers.values(),
).T
what_if.index.name = "credit score"
what_if.map("{:.0%}".format)

,Starter,Standard,Premium,Elite
credit score,,,,
500,25%,10%,3%,1%
550,49%,24%,7%,3%
600,76%,50%,19%,7%
650,91%,75%,44%,16%
700,96%,90%,71%,42%
750,98%,96%,88%,68%
800,99%,98%,95%,84%
850,100%,99%,98%,93%


Same applicant (score 700, \$60k income, Standard card), different debt levels:

In [9]:
debts = [0, 5_000, 15_000, 30_000, 50_000, 80_000]
pd.Series(
    {f"${d:,}": predictor.predict(700, 60_000, d, card_tiers=[1])[0] for d in debts},
    name="approval probability",
).map("{:.0%}".format)

$0         95%
$5,000     93%
$15,000    89%
$30,000    71%
$50,000    44%
$80,000    21%
Name: approval probability, dtype: str

## 5. The tensor-shape gotcha

`nn.Linear` expects a 2D tensor shaped `[batch, features]`. One applicant's features form a 1D tensor of shape `[4]`, so we must add the batch dimension to get `[1, 4]`. `ApprovalPredictor.to_tensor()` does this automatically and rejects the wrong number of features.

In [10]:
one_row = build_features(700, 60_000, 12_000, 1)
print("raw features:     ", torch.tensor(one_row).shape)
print("after to_tensor():", ApprovalPredictor.to_tensor(one_row).shape)

try:
    ApprovalPredictor.to_tensor([[700, 60_000, 12_000]])  # forgot the card tier
except ValueError as err:
    print("rejected:         ", err)

raw features:      torch.Size([4])
after to_tensor(): torch.Size([1, 4])
rejected:          Expected a tensor of shape [batch, 4], got [1, 3]
